# Notebook 05 — Mediation Network Analysis
## Generalised Metabolite–Metagenomics Correlation Pipeline

**Inputs:**
- `analysis_results.pkl` (NB02): `spe`, `mtb`, `meta_yk`, `nm_y`, `corr_all`, `corr_partial_sig`
- `ml_results.pkl` (NB03): `shap_summary`, `benchmark`, `targets`, `valid_mask`

**Analyses:**
1. Bipartite species–metabolite network from partial-correlation pairs
2. Network visualisation (hub-only spring layout)
3. KEGG pathway enrichment (REST API + local fallback)
4. Bootstrap mediation ACME (1000 iterations; top SHAP-ranked pairs)
5. Mediation forest plot (95% CI; BH-corrected permutation p-values)
6. Network centrality analysis (degree + betweenness; scatter plot)

**Mediation model direction:** `X = Stage.Num → M = species CLR → Y = metabolite log10`
Tests whether *species abundance mediates the stage→metabolite relationship*.
Note: the reverse direction (`X = species → M = metabolite → Y = stage`) is biologically
motivated (species produce metabolites that drive CRC) and should be compared in future work.

**Output:** `mediation_results.pkl`

## 1 · Imports & Setup

In [1]:
import sys, warnings, time, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

NB_DIR = Path(".").resolve()
sys.path.insert(0, str(NB_DIR))

from utils import (
    INTER_DIR, FIG_DIR, TABLE_DIR,
    FDR_THRESHOLD, MIN_CORR,
    PALETTE_3GROUP,
    annotate_pathway, extract_genus, pathway_kegg_ids,
    bootstrap_mediation,
    load_pickle, save_pickle, savefig,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.1)

# ── tunable parameters ──────────────────────────────────────────────────────
N_MEDIATION_PAIRS = 30   # top SHAP pairs to run bootstrap mediation on
MIN_HUB_DEGREE    = 3    # min edges for hub-subgraph visualisation
BOOT_ITER         = 1000 # bootstrap iterations
KEGG_PAUSE_S      = 0.35 # polite delay between KEGG REST API calls
# ────────────────────────────────────────────────────────────────────────────

for d in [FIG_DIR / "network", TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete.")

Setup complete.


## 2 · Load Data

In [2]:
# ── NB02 results ─────────────────────────────────────────────────────────────
ana = load_pickle(INTER_DIR / "analysis_results.pkl")

# Keys match what NB02 saves: spe, mtb, meta_yk, nm_y, corr_all, corr_partial_sig
sp_clr  = ana["spe"]       # samples × species  (CLR-transformed)
mt_log  = ana["mtb"]       # samples × metabolites (log10-transformed)
meta_y  = ana["meta_yk"]   # harmonised metadata
nm_y    = ana.get("nm_y", {})  # KEGG ID → metabolite name map
nm_y = {k: (v if isinstance(v, str) else k) for k, v in nm_y.items()}

# Stage.Num guard — reconstruct if missing (requires Stage.6)
if "Stage.Num" not in meta_y.columns:
    from utils import CRC_STAGE_NUMERIC
    meta_y = meta_y.copy()
    meta_y["Stage.Num"] = meta_y.get("Stage.6", meta_y.get("study_group", pd.Series(dtype=str))).map(CRC_STAGE_NUMERIC)
    print("Stage.Num reconstructed from Stage.6.")

# Use partial correlations when available; fall back to global Spearman
corr_partial = ana.get("corr_partial_sig", pd.DataFrame())
corr_global  = ana.get("corr_all", pd.DataFrame())   # NB02 saves as "corr_all"

if not corr_partial.empty and "rho_partial" in corr_partial.columns:
    corr_net = corr_partial.copy()
    corr_net["rho_use"] = corr_net["rho_partial"]
    print("Using partial-correlation pairs.")
elif not corr_partial.empty:
    corr_net = corr_partial.copy()
    corr_net["rho_use"] = corr_net["rho"]
    print("Partial-corr available but no rho_partial column — using rho.")
else:
    corr_net = corr_global.copy()
    corr_net["rho_use"] = corr_net["rho"]
    print("Partial-corr not available — falling back to global Spearman.")

print(f"Correlation network edges (before q-filter): {len(corr_net)}")

# ── NB03 results ─────────────────────────────────────────────────────────────
ml = load_pickle(INTER_DIR / "ml_results.pkl")

shap_summary = ml.get("shap_summary", pd.DataFrame())
targets      = ml.get("targets", [])
benchmark    = ml.get("benchmark", pd.DataFrame())
valid_mask   = ml.get("valid_mask", None)

# Validate SHAP summary orientation: expected rows=metabolites, cols=species
if not shap_summary.empty and len(targets) > 0:
    if len(shap_summary) > len(shap_summary.columns) * 10:
        print("WARNING: SHAP summary may be transposed (more rows than cols). "
              f"Shape: {shap_summary.shape}, targets: {len(targets)}")

print(f"SHAP summary shape : {shap_summary.shape}")
print(f"ML targets loaded  : {len(targets)}")

Loaded: C:\Users\andna\Documents\KI\Results\intermediate\analysis_results.pkl
Using partial-correlation pairs.
Correlation network edges (before q-filter): 1997
Loaded: C:\Users\andna\Documents\KI\Results\intermediate\ml_results.pkl
SHAP summary shape : (49, 372)
ML targets loaded  : 50


## 3 · Build Bipartite Species–Metabolite Network

In [3]:
# filter to FDR-significant edges
net_edges = corr_net[corr_net["qval"] <= FDR_THRESHOLD].copy()
print(f"Network edges (q ≤ {FDR_THRESHOLD}): {len(net_edges)}")

G = nx.Graph()

for _, row in net_edges.iterrows():
    spe  = row["species"]
    mtb  = row["metabolite"]
    rho  = float(row["rho_use"])
    qval = float(row["qval"])
    G.add_node(spe, node_type="species")
    G.add_node(mtb, node_type="metabolite")
    G.add_edge(spe, mtb, rho=rho, qval=qval, weight=abs(rho))

species_nodes    = [n for n, d in G.nodes(data=True) if d.get("node_type") == "species"]
metabolite_nodes = [n for n, d in G.nodes(data=True) if d.get("node_type") == "metabolite"]

print(f"Graph: {G.number_of_nodes()} nodes | "
      f"{len(species_nodes)} species, {len(metabolite_nodes)} metabolites | "
      f"{G.number_of_edges()} edges")

Network edges (q ≤ 0.05): 1997
Graph: 334 nodes | 223 species, 111 metabolites | 1997 edges


## 4 · Network Visualisation (Hub Nodes)

In [4]:
# extract hub subgraph
hub_nodes = [n for n, d in G.degree() if d >= MIN_HUB_DEGREE]
G_hub     = G.subgraph(hub_nodes).copy() if len(hub_nodes) >= 6 else G.copy()
print(f"Hub subgraph: {G_hub.number_of_nodes()} nodes, "
      f"{G_hub.number_of_edges()} edges  (min degree {MIN_HUB_DEGREE})")

pos = nx.spring_layout(G_hub, seed=42, k=0.6)

pw_colors = {
    "Polyamine":          "#E91E63",
    "SCFA":               "#4CAF50",
    "Bile Acid":          "#FF9800",
    "Amino Acid":         "#9C27B0",
    "Lipid / Fatty Acid": "#795548",
    "Indole / Tryptophan":"#00BCD4",
    "Nucleotide":         "#607D8B",
}

node_colors, node_sizes = [], []
for n in G_hub.nodes():
    ntype = G_hub.nodes[n].get("node_type", "species")
    deg   = G_hub.degree(n)
    if ntype == "species":
        node_colors.append("#1976D2")
        node_sizes.append(80 + deg * 25)
    else:
        pw = annotate_pathway(n)
        node_colors.append(pw_colors.get(pw, "#78909C"))
        node_sizes.append(150 + deg * 20)

edge_colors = ["#E53935" if G_hub[u][v]["rho"] > 0 else "#1565C0"
               for u, v in G_hub.edges()]
edge_widths = [0.8 + abs(G_hub[u][v]["rho"]) * 2.5 for u, v in G_hub.edges()]

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_edges(G_hub, pos, ax=ax,
                       edge_color=edge_colors, width=edge_widths, alpha=0.50)
nx.draw_networkx_nodes(G_hub, pos, ax=ax,
                       node_color=node_colors, node_size=node_sizes, alpha=0.88)

# label top-20 by degree
top_deg = sorted(G_hub.degree(), key=lambda x: x[1], reverse=True)[:20]
label_dict = {n: n.split("__")[-1][:26] for n, _ in top_deg}
nx.draw_networkx_labels(G_hub, pos, labels=label_dict,
                        ax=ax, font_size=6.5, font_color="#212121")

legend_handles = [
    mpatches.Patch(color="#1976D2",  label="Species"),
    mpatches.Patch(color="#E91E63",  label="Polyamine"),
    mpatches.Patch(color="#4CAF50",  label="SCFA"),
    mpatches.Patch(color="#FF9800",  label="Bile Acid"),
    mpatches.Patch(color="#9C27B0",  label="Amino Acid"),
    mpatches.Patch(color="#78909C",  label="Other metabolite"),
    plt.Line2D([0],[0], color="#E53935", lw=2, label="Positive ρ"),
    plt.Line2D([0],[0], color="#1565C0", lw=2, label="Negative ρ"),
]
ax.legend(handles=legend_handles, loc="upper left", fontsize=8, framealpha=0.9)
ax.set_title("Bipartite Species–Metabolite Network (Hub Nodes)",
             fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
savefig(fig, "network", "nb05_bipartite_network_hub.png")

Hub subgraph: 228 nodes, 1861 edges  (min degree 3)
Saved figure: C:\Users\andna\Documents\KI\Results\figures\network\nb05_bipartite_network_hub.png


## 5 · KEGG Pathway Enrichment

In [5]:
# annotate metabolite nodes with pathway membership
mtb_in_net = [n for n in G.nodes() if G.nodes[n].get("node_type") == "metabolite"]
pathway_annot = {m: annotate_pathway(m) for m in mtb_in_net}

# KEGG REST API: fetch names for unannotated KEGG compound IDs
def _kegg_name(kegg_id: str, pause: float = KEGG_PAUSE_S) -> str | None:
    url = f"https://rest.kegg.jp/get/{kegg_id}"
    try:
        with urllib.request.urlopen(url, timeout=8) as r:
            for line in r.read().decode().splitlines():
                if line.startswith("NAME"):
                    return line.split(None, 1)[1].strip().rstrip(";")
    except Exception:
        pass
    finally:
        time.sleep(pause)
    return None

def _bare_kegg(mid: str) -> str:
    """Extract bare KEGG compound ID from 'C00134_Putrescine' format."""
    return mid.split("_")[0] if "_" in mid else mid

unannotated = [
    m for m in mtb_in_net
    if pathway_annot.get(m) == "Other"
    and _bare_kegg(m).startswith("C") and len(_bare_kegg(m)) == 6
][:40]  # cap API calls
print(f"Querying KEGG REST for {len(unannotated)} unannotated compound IDs …")
kegg_names: dict[str, str] = {}
for kid_full in unannotated:
    name = _kegg_name(_bare_kegg(kid_full))
    if name:
        kegg_names[kid_full] = name

# Fisher's exact enrichment: is each pathway over-represented in the network?
all_mtb_bg = set(mt_log.columns)
n_bg       = len(all_mtb_bg)
n_net      = len(mtb_in_net)

enrich_rows = []
for pw in ["Polyamine", "SCFA", "Bile Acid", "Amino Acid",
           "Lipid / Fatty Acid", "Indole / Tryptophan", "Nucleotide"]:
    pw_ids = set(pathway_kegg_ids(pw).keys())
    k_net  = len({m for m in mtb_in_net if _bare_kegg(m) in pw_ids})
    k_bg   = len({m for m in all_mtb_bg if _bare_kegg(m) in pw_ids})
    table  = [
        [k_net,          n_net - k_net],
        [k_bg - k_net,   (n_bg - n_net) - (k_bg - k_net)],
    ]
    if all(v >= 0 for row in table for v in row):
        odds, pval = fisher_exact(table, alternative="greater")
    else:
        odds, pval = np.nan, np.nan

    enrich_rows.append({
        "pathway":      pw,
        "k_network":    k_net,
        "k_background": k_bg,
        "n_network":    n_net,
        "n_background": n_bg,
        "odds_ratio":   float(odds) if not np.isnan(odds) else np.nan,
        "pval":         float(pval) if not np.isnan(pval) else np.nan,
    })

enrich_df = pd.DataFrame(enrich_rows)
valid_p = enrich_df["pval"].notna()
if valid_p.any():
    _, qvals, _, _ = multipletests(enrich_df.loc[valid_p, "pval"], method="fdr_bh")
    enrich_df.loc[valid_p, "qval"] = qvals

print("\nKEGG Pathway Enrichment (Fisher exact):")
print(enrich_df.to_string(index=False))
enrich_df.to_csv(TABLE_DIR / "kegg_pathway_enrichment.csv", index=False)

Querying KEGG REST for 40 unannotated compound IDs …

KEGG Pathway Enrichment (Fisher exact):
            pathway  k_network  k_background  n_network  n_background  odds_ratio     pval     qval
          Polyamine          2             2        111           150         inf 0.546309 0.863333
               SCFA          1             1        111           150         inf 0.740000 0.863333
          Bile Acid          2             2        111           150         inf 0.546309 0.863333
         Amino Acid          2             2        111           150         inf 0.546309 0.863333
 Lipid / Fatty Acid          0             0        111           150         NaN 1.000000 1.000000
Indole / Tryptophan          1             1        111           150         inf 0.740000 0.863333
         Nucleotide          1             1        111           150         inf 0.740000 0.863333


## 6 · Bootstrap Mediation ACME

> **Model direction note (S4):** The current causal model is
> **X = Stage.Num → M = species CLR → Y = metabolite log₁₀**, testing whether microbial
> abundance mediates the stage→metabolite relationship. Biologically, the more causal
> direction is **X = species → M = metabolite → Y = Stage** (species produce metabolites
> that drive CRC progression). Both directions are scientifically defensible; the chosen
> direction is standard for stage-stratified microbiome studies. For the publication
> discussion, acknowledge both directions and note that the reverse model (species as
> exposure) should be run as a sensitivity analysis.

In [6]:
mediation_df = pd.DataFrame()  # initialise — prevents NameError in save cell

if shap_summary.empty:
    print("SHAP summary empty — falling back to top partial-correlation pairs.")
    # Build fake shap_long from network edges sorted by |rho|
    shap_long = net_edges[["species", "metabolite", "rho_use"]].copy()
    shap_long = shap_long.rename(columns={"rho_use": "shap_imp"})
    shap_long["shap_imp"] = shap_long["shap_imp"].abs()
    shap_long = shap_long.sort_values("shap_imp", ascending=False)
else:
    # melt SHAP summary: rows = metabolites, cols = species
    shap_long = (
        shap_summary.stack()
        .reset_index()
        .rename(columns={"level_0": "metabolite", "level_1": "species", 0: "shap_imp"})
    )
    shap_long = shap_long[shap_long["shap_imp"] > 0].sort_values("shap_imp", ascending=False)

# prefer pairs that also appear in the correlation network
net_pair_set = set(zip(net_edges["species"], net_edges["metabolite"]))
shap_filt = shap_long[
    shap_long.apply(lambda r: (r["species"], r["metabolite"]) in net_pair_set, axis=1)
].head(N_MEDIATION_PAIRS)
if len(shap_filt) < 5:
    shap_filt = shap_long.head(N_MEDIATION_PAIRS)
    print("  Network filter relaxed — using top SHAP pairs regardless of network membership.")

print(f"Running bootstrap mediation on {len(shap_filt)} pairs "
      f"(n_boot={BOOT_ITER}) …")

# sample alignment — use NB03 valid_mask if available
common_idx = sp_clr.index.intersection(mt_log.index).intersection(meta_y.index)
if valid_mask is not None:
    common_idx = common_idx.intersection(meta_y.index[valid_mask])

sp_sub     = sp_clr.loc[common_idx]
mt_sub     = mt_log.loc[common_idx]
stage_x    = meta_y.loc[common_idx, "Stage.Num"].values.astype(float)

med_rows = []
for i, (_, pair) in enumerate(shap_filt.iterrows()):
    spe      = pair["species"]
    mtb      = pair["metabolite"]
    shap_imp = float(pair["shap_imp"])

    if spe not in sp_sub.columns or mtb not in mt_sub.columns:
        continue

    # X = Stage.Num (exposure), M = species CLR (mediator), Y = metabolite log10 (outcome)
    med = bootstrap_mediation(
        x=stage_x,
        m=sp_sub[spe].values,
        y=mt_sub[mtb].values,
        n_boot=BOOT_ITER,
        random_state=42 + i,
    )
    row = {
        "species":           spe,
        "metabolite":        mtb,
        "metabolite_name":   nm_y.get(mtb, mtb),
        "shap_imp":          shap_imp,
        "pathway":           annotate_pathway(mtb),
    }
    row.update(med)
    med_rows.append(row)

    if (i + 1) % 10 == 0:
        print(f"  {i + 1} / {len(shap_filt)} done")

mediation_df = pd.DataFrame(med_rows)

# BH-correct permutation p-values
if not mediation_df.empty and "p_value" in mediation_df.columns:
    valid_p = mediation_df["p_value"].notna()
    if valid_p.sum() > 1:
        _, qvals, _, _ = multipletests(
            mediation_df.loc[valid_p, "p_value"], method="fdr_bh")
        mediation_df.loc[valid_p, "q_value"] = qvals
    mediation_df = mediation_df.sort_values("acme", key=abs, ascending=False)

n_sig = int((mediation_df["q_value"] <= FDR_THRESHOLD).sum()) \
    if "q_value" in mediation_df.columns else 0
print(f"\nMediation complete. {len(mediation_df)} pairs tested, "
      f"{n_sig} significant (q ≤ {FDR_THRESHOLD}).")
mediation_df.to_csv(TABLE_DIR / "bootstrap_mediation_results.csv", index=False)

Running bootstrap mediation on 30 pairs (n_boot=1000) …
  10 / 30 done
  20 / 30 done
  30 / 30 done

Mediation complete. 30 pairs tested, 30 significant (q ≤ 0.05).


## 7 · Mediation Forest Plot

In [7]:
if mediation_df.empty:
    print("No mediation results — skipping forest plot.")
else:
    plot_df = (
        mediation_df
        .dropna(subset=["acme", "acme_ci_lo", "acme_ci_hi"])
        .sort_values("acme", ascending=True)
        .reset_index(drop=True)
    )

    plot_df["label"] = [
        f"{r.species.split('__')[-1][:20]} → {r.metabolite_name[:22]}"
        for _, r in plot_df.iterrows()
    ]

    if "q_value" in plot_df.columns:
        sig_mask = plot_df["q_value"] <= FDR_THRESHOLD
    else:
        sig_mask = plot_df["p_value"] <= 0.05 if "p_value" in plot_df.columns \
                   else pd.Series(False, index=plot_df.index)

    colors = np.where(sig_mask, "#D32F2F", "#90A4AE")

    fig, ax = plt.subplots(figsize=(9, max(4, len(plot_df) * 0.40)))
    y_pos = np.arange(len(plot_df))

    # errorbar does not accept per-point color arrays — loop over rows
    for j in range(len(plot_df)):
        row = plot_df.iloc[j]
        xerr_lo = max(0, row["acme"] - row["acme_ci_lo"])
        xerr_hi = max(0, row["acme_ci_hi"] - row["acme"])
        ax.errorbar(
            x=row["acme"], y=y_pos[j],
            xerr=[[xerr_lo], [xerr_hi]],
            fmt="o", color=colors[j], ecolor=colors[j],
            capsize=3, markersize=5, linewidth=1.0,
        )

    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_df["label"], fontsize=7)
    ax.set_xlabel("ACME  (bootstrap 95 % CI)", fontsize=10)
    ax.set_title("Bootstrap Mediation — Species → Metabolite via CRC Stage",
                 fontsize=11, fontweight="bold")

    sig_patch = mpatches.Patch(color="#D32F2F", label=f"q ≤ {FDR_THRESHOLD}")
    ns_patch  = mpatches.Patch(color="#90A4AE", label="Not significant")
    ax.legend(handles=[sig_patch, ns_patch], fontsize=8, loc="lower right")

    plt.tight_layout()
    savefig(fig, "network", "nb05_mediation_forest.png")

Saved figure: C:\Users\andna\Documents\KI\Results\figures\network\nb05_mediation_forest.png


## 8 · Network Centrality Analysis

In [8]:
print("Computing degree centrality …")
deg_cent = nx.degree_centrality(G)

print("Computing betweenness centrality …")
# approximate for large graphs
if G.number_of_nodes() > 500:
    k_approx = min(200, G.number_of_nodes())
    bet_cent = nx.betweenness_centrality(G, k=k_approx, normalized=True, seed=42)
else:
    bet_cent = nx.betweenness_centrality(G, normalized=True)

centrality_df = pd.DataFrame({
    "node":         list(deg_cent.keys()),
    "node_type":    [G.nodes[n].get("node_type", "unknown") for n in deg_cent],
    "degree_raw":   [G.degree(n) for n in deg_cent],
    "degree_cent":  [deg_cent[n] for n in deg_cent],
    "between_cent": [bet_cent[n] for n in deg_cent],
})
centrality_df = centrality_df.sort_values("between_cent", ascending=False).reset_index(drop=True)

print("Top 10 bridging nodes (betweenness centrality):")
print(centrality_df.head(10)[["node", "node_type", "degree_raw", "between_cent"]].to_string(index=False))

centrality_df.to_csv(TABLE_DIR / "network_centrality.csv", index=False)

Computing degree centrality …
Computing betweenness centrality …
Top 10 bridging nodes (betweenness centrality):
                     node  node_type  degree_raw  between_cent
    C02678_Dodecanedioate metabolite         105      0.100507
        C00134_Putrescine metabolite          99      0.089976
          C08277_Sebacate metabolite         104      0.085940
 C01042_N-Acetylaspartate metabolite          35      0.076032
C02714_N-Acetylputrescine metabolite          98      0.068812
          C02656_Pimelate metabolite          92      0.068770
   C00431_5-Aminovalerate metabolite          84      0.053017
        C00398_Tryptamine metabolite          47      0.049684
      CAG-170 sp000432135    species          35      0.048926
         Alistipes shahii    species          43      0.046875


In [9]:
fig, ax = plt.subplots(figsize=(9, 7))

for ntype, color, marker in [
    ("species",    "#1976D2", "o"),
    ("metabolite", "#E91E63", "s"),
]:
    sub = centrality_df[centrality_df["node_type"] == ntype]
    ax.scatter(sub["degree_cent"], sub["between_cent"],
               c=color, marker=marker, alpha=0.65, s=50, label=ntype.capitalize())

# label top-15 bridging nodes
for _, row in centrality_df.head(15).iterrows():
    short = row["node"].split("__")[-1][:22]
    ax.annotate(short,
                xy=(row["degree_cent"], row["between_cent"]),
                xytext=(4, 4), textcoords="offset points",
                fontsize=7, color="#37474F")

ax.set_xlabel("Degree Centrality",      fontsize=11)
ax.set_ylabel("Betweenness Centrality", fontsize=11)
ax.set_title("Network Centrality — Species & Metabolite Nodes",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
savefig(fig, "network", "nb05_centrality_scatter.png")

Saved figure: C:\Users\andna\Documents\KI\Results\figures\network\nb05_centrality_scatter.png


## 9 · Save Results

In [10]:
mediation_results = {
    "network":        G,
    "net_edges":      net_edges,
    "mediation_df":   mediation_df,
    "enrich_df":      enrich_df,
    "kegg_names":     kegg_names,
    "centrality_df":  centrality_df,
    "pathway_annot":  pathway_annot,
}

save_pickle(mediation_results, INTER_DIR / "mediation_results.pkl")

print("\n=== NB05 complete ===")
print(f"  Network edges     : {len(net_edges)}")
print(f"  Mediation pairs   : {len(mediation_df)}")
print(f"  Enriched pathways : {len(enrich_df)}")
print(f"  Centrality nodes  : {len(centrality_df)}")
print("Run NB06 (GutSMASH benchmarking) next.")

Saved: C:\Users\andna\Documents\KI\Results\intermediate\mediation_results.pkl

=== NB05 complete ===
  Network edges     : 1997
  Mediation pairs   : 30
  Enriched pathways : 7
  Centrality nodes  : 334
Run NB06 (GutSMASH benchmarking) next.
